# Bureau Data Cleaning

This notebook cleans the bureau credit records using the problems found during EDA. It keeps records linked to the project applicants, checks the IDs, groups rare text values, corrects invalid dates and removes features with too many missing values.

## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "bureau.csv"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "bureau_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/bureau.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/bureau_clean.pkl


## Load the bureau data and applicant IDs

The bureau file also contains records for applicants outside the labelled application data. Only records linked to the training or test applicants are needed for this project.

In [7]:
bureau_raw = pd.read_csv(raw_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
project_ids = set(training_ids).union(set(test_ids))
original_rows, original_columns = bureau_raw.shape

bureau_clean = bureau_raw.loc[bureau_raw["SK_ID_CURR"].isin(project_ids)].copy().reset_index(drop=True)
out_of_scope_rows = original_rows - len(bureau_clean)
training_record_mask = bureau_clean["SK_ID_CURR"].isin(set(training_ids))
training_bureau = bureau_clean.loc[training_record_mask].copy()

print("Raw bureau rows:", original_rows)
print("Project bureau rows retained:", len(bureau_clean))
print("Rows not linked to the labelled dataset:", out_of_scope_rows)
print("Bureau rows linked to the training set:", len(training_bureau))
print("Project applicants with bureau history:", bureau_clean["SK_ID_CURR"].nunique())

Raw bureau rows: 1716428
Project bureau rows retained: 1465325
Rows not linked to the labelled dataset: 251103
Bureau rows linked to the training set: 1171942
Project applicants with bureau history: 263491


The rows that were removed belong to applicant IDs outside the labelled application data, so they are not needed for this project.


## Check IDs and duplicate records

In [10]:
missing_current_ids = int(bureau_clean["SK_ID_CURR"].isna().sum())
missing_bureau_ids = int(bureau_clean["SK_ID_BUREAU"].isna().sum())
duplicate_bureau_ids = int(bureau_clean.duplicated("SK_ID_BUREAU").sum())
exact_duplicates = int(bureau_clean.duplicated().sum())

if exact_duplicates > 0:
    bureau_clean = bureau_clean.drop_duplicates().reset_index(drop=True)

assert missing_current_ids == 0
assert missing_bureau_ids == 0
assert bureau_clean["SK_ID_BUREAU"].is_unique, "SK_ID_BUREAU must remain unique for bureau-balance merging."
print("Missing SK_ID_CURR:", missing_current_ids)
print("Missing SK_ID_BUREAU:", missing_bureau_ids)
print("Duplicate bureau IDs before cleaning:", duplicate_bureau_ids)
print("Exact duplicate rows removed:", exact_duplicates)

Missing SK_ID_CURR: 0
Missing SK_ID_BUREAU: 0
Duplicate bureau IDs before cleaning: 0
Exact duplicate rows removed: 0


The bureau IDs are clean, no duplicates or missing values. They can be used later to connect this table with the bureau-balance data.


## Clean and group text values

Text values are checked for extra spaces, and missing text values are changed to `Unknown`. Almost all records use `currency 1`, so the other rare currencies are combined into one group.

Credit types with fewer than 100 training records are also combined into an `Other rare credit` group. This avoids keeping separate categories with very little data.

In [13]:
categorical_columns = bureau_clean.select_dtypes(exclude="number").columns.tolist()
categorical_missing_count = int(bureau_clean[categorical_columns].isna().sum().sum())

for column in categorical_columns:
    bureau_clean[column] = bureau_clean[column].str.strip()
    bureau_clean[column] = bureau_clean[column].fillna("Unknown")

bureau_clean["CREDIT_CURRENCY"] = bureau_clean["CREDIT_CURRENCY"].where(
    bureau_clean["CREDIT_CURRENCY"].eq("currency 1"), "Other currency"
)
credit_type_counts = training_bureau["CREDIT_TYPE"].value_counts()
rare_credit_types = credit_type_counts[credit_type_counts < 100].index
bureau_clean["CREDIT_TYPE"] = bureau_clean["CREDIT_TYPE"].where(
    ~bureau_clean["CREDIT_TYPE"].isin(rare_credit_types), "Other rare credit"
)
print("Rare credit types consolidated:", len(rare_credit_types))
print("Missing text values changed to Unknown:", categorical_missing_count)
print(bureau_clean["CREDIT_CURRENCY"].value_counts())

Rare credit types consolidated: 6
Missing text values changed to Unknown: 0
CREDIT_CURRENCY
currency 1        1464094
Other currency       1231
Name: count, dtype: int64


Six rare credit types were combined into one group. Almost all bureau records use currency 1, so the rest were grouped too.


## Correct invalid date values

The bureau date columns are stored as days before the current application. Positive values and extremely large date codes do not follow the usual meaning of these columns.

These values are changed to missing. A new column also records which bureau rows originally contained an unusual date.

In [16]:
future_credit_start = bureau_clean["DAYS_CREDIT"] > 0
future_actual_end = bureau_clean["DAYS_ENDDATE_FACT"] > 0
future_update = bureau_clean["DAYS_CREDIT_UPDATE"] > 0
extreme_enddate = bureau_clean["DAYS_CREDIT_ENDDATE"].abs() > 36500
extreme_actual_end = bureau_clean["DAYS_ENDDATE_FACT"].abs() > 36500
extreme_update = bureau_clean["DAYS_CREDIT_UPDATE"].abs() > 36500

bureau_clean["BUREAU_DATE_ANOMALY"] = (future_credit_start | future_actual_end | future_update | extreme_enddate | extreme_actual_end | extreme_update).astype("int8")
bureau_clean.loc[future_credit_start, "DAYS_CREDIT"] = np.nan
bureau_clean.loc[future_actual_end | extreme_actual_end, "DAYS_ENDDATE_FACT"] = np.nan
bureau_clean.loc[future_update | extreme_update, "DAYS_CREDIT_UPDATE"] = np.nan
bureau_clean.loc[extreme_enddate, "DAYS_CREDIT_ENDDATE"] = np.nan

print("Future credit starts corrected:", int(future_credit_start.sum()))
print("Future actual end dates corrected:", int(future_actual_end.sum()))
print("Future update dates corrected:", int(future_update.sum()))
print("Extreme date codes corrected:", int((extreme_enddate | extreme_actual_end | extreme_update).sum()))

Future credit starts corrected: 0
Future actual end dates corrected: 0
Future update dates corrected: 17
Extreme date codes corrected: 230


17 future update dates and 230 extreme date records were corrected. The original rows were kept, just the invalid date values were changed to missing.


## Check financial values

Some financial values look unusual, but they are not automatically errors. Negative debt may represent an overpayment or correction, and debt above the original credit amount may include interest or reporting changes.

These records are kept. New indicator columns are created so that the later feature-engineering step can record where the unusual values appeared.

In [19]:
negative_debt = bureau_clean["AMT_CREDIT_SUM_DEBT"] < 0
negative_limit = bureau_clean["AMT_CREDIT_SUM_LIMIT"] < 0
debt_above_credit = bureau_clean["AMT_CREDIT_SUM_DEBT"] > bureau_clean["AMT_CREDIT_SUM"]
overdue_without_amount = bureau_clean["CREDIT_DAY_OVERDUE"].gt(0) & bureau_clean["AMT_CREDIT_SUM_OVERDUE"].eq(0)

bureau_clean["BUREAU_NEGATIVE_DEBT"] = negative_debt.fillna(False).astype("int8")
bureau_clean["BUREAU_NEGATIVE_LIMIT"] = negative_limit.fillna(False).astype("int8")
bureau_clean["BUREAU_DEBT_ABOVE_CREDIT"] = debt_above_credit.fillna(False).astype("int8")
bureau_clean["BUREAU_OVERDUE_INCONSISTENCY"] = overdue_without_amount.fillna(False).astype("int8")

print("Negative debt records retained:", int(negative_debt.sum()))
print("Negative limit records retained:", int(negative_limit.sum()))
print("Debt above credit records:", int(debt_above_credit.sum()))
print("Overdue-day/amount inconsistencies:", int(overdue_without_amount.sum()))

Negative debt records retained: 8418
Negative limit records retained: 351
Debt above credit records: 25029
Overdue-day/amount inconsistencies: 274


Some negative debt values, negative credit limits, debt higher than the credit amount, and mismatched overdue days/amounts were found. These were kept and flagged instead of being deleted.


## Select features using the training set

The missing-value rules are calculated using only bureau records linked to the training set. This keeps the test set separate from the cleaning decisions.

A bureau feature is removed when at least 50% of its training-linked values are missing or when it has only one value.

Target correlation is not used at this stage because one applicant can have several bureau records. Correlation with default will be checked later, after the records have been summarized into one row per applicant.

In [22]:
MISSING_THRESHOLD = 0.50
training_record_mask = bureau_clean["SK_ID_CURR"].isin(set(training_ids))
training_bureau = bureau_clean.loc[training_record_mask]
decision_rows = []

for column in bureau_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_BUREAU"]:
        continue
    series = training_bureau[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Kept for applicant-level summaries and later feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"At least {MISSING_THRESHOLD:.0%} missing in training-linked bureau records"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "The same value appears in all training-linked bureau records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision, "reason": reason,
        "target_association_stage": "Checked later after creating one row per applicant",
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(["decision", "missing_rate"], ascending=[True, False]).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
bureau_clean = bureau_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['AMT_ANNUITY', 'AMT_CREDIT_MAX_OVERDUE']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,DAYS_ENDDATE_FACT,float64,435850,0.37190,2913,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
1,AMT_CREDIT_SUM_LIMIT,float64,391802,0.33432,36583,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
2,AMT_CREDIT_SUM_DEBT,float64,178498,0.15231,173792,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
3,DAYS_CREDIT_ENDDATE,float64,71374,0.06090,13211,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
4,DAYS_CREDIT_UPDATE,float64,85,0.00007,2903,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
5,AMT_CREDIT_SUM,float64,3,0.00000,181432,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
6,CREDIT_ACTIVE,str,0,0.00000,4,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
7,CREDIT_CURRENCY,str,0,0.00000,2,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
8,DAYS_CREDIT,float64,0,0.00000,2923,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...
9,CREDIT_DAY_OVERDUE,int64,0,0.00000,764,Keep,Kept for applicant-level summaries and later f...,Checked later after creating one row per appli...


Two features were removed: AMT_ANNUITY (about 77% missing) and AMT_CREDIT_MAX_OVERDUE (about 65% missing). The rest were kept since their missing rates were below 50%. Their remaining missing values will be handled later during feature creation and modelling.


## Create missing-value measures

In [25]:
record_feature_columns = [
    column for column in bureau_clean.columns
    if column not in ["SK_ID_CURR", "SK_ID_BUREAU"]
]

bureau_clean["BUREAU_RECORD_MISSING_COUNT"] = (
    bureau_clean[record_feature_columns].isna().sum(axis=1).astype("int8")
)

bureau_clean["BUREAU_RECORD_MISSING_RATE"] = (
    bureau_clean["BUREAU_RECORD_MISSING_COUNT"] / len(record_feature_columns)
)

print(
    bureau_clean[
        ["BUREAU_RECORD_MISSING_COUNT", "BUREAU_RECORD_MISSING_RATE"]
    ].describe().round(4)
)

       BUREAU_RECORD_MISSING_COUNT  BUREAU_RECORD_MISSING_RATE
count                 1.465325e+06                1.465325e+06
mean                  9.191000e-01                5.110000e-02
std                   9.176000e-01                5.100000e-02
min                   0.000000e+00                0.000000e+00
25%                   0.000000e+00                0.000000e+00
50%                   1.000000e+00                5.560000e-02
75%                   1.000000e+00                5.560000e-02
max                   4.000000e+00                2.222000e-01


These two columns show how much information is missing for each bureau record. On average, a record is missing about 5% of its fields, with the highest around 22%. This could be useful later, since applicants with more missing bureau information might behave differently in terms of default risk.


## Check the cleaned bureau data

In [28]:
numeric_clean = bureau_clean.select_dtypes(include="number")
infinite_count = int(np.isinf(numeric_clean.to_numpy()).sum())
retained_project_ids = set(bureau_clean["SK_ID_CURR"])
validation_checks = pd.DataFrame([
    {"check": "Only project applicant records retained", "passed": retained_project_ids.issubset(project_ids)},
    {"check": "SK_ID_CURR complete", "passed": bureau_clean["SK_ID_CURR"].notna().all()},
    {"check": "SK_ID_BUREAU complete", "passed": bureau_clean["SK_ID_BUREAU"].notna().all()},
    {"check": "SK_ID_BUREAU unique", "passed": bureau_clean["SK_ID_BUREAU"].is_unique},
    {"check": "No exact duplicates", "passed": not bureau_clean.duplicated().any()},
    {"check": "No categorical missing values", "passed": bureau_clean.select_dtypes(exclude="number").isna().sum().sum() == 0},
    {"check": "No future credit starts", "passed": not bureau_clean["DAYS_CREDIT"].gt(0).any()},
    {"check": "No future actual end dates", "passed": not bureau_clean["DAYS_ENDDATE_FACT"].gt(0).any()},
    {"check": "No future credit updates", "passed": not bureau_clean["DAYS_CREDIT_UPDATE"].gt(0).any()},
    {"check": "High-missing AMT_ANNUITY removed", "passed": "AMT_ANNUITY" not in bureau_clean.columns},
    {"check": "High-missing maximum-overdue feature removed", "passed": "AMT_CREDIT_MAX_OVERDUE" not in bureau_clean.columns},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one bureau cleaning check failed."
validation_checks

,check,passed
0,Only project applicant records retained,True
1,SK_ID_CURR complete,True
2,SK_ID_BUREAU complete,True
3,SK_ID_BUREAU unique,True
4,No exact duplicates,True
5,No categorical missing values,True
6,No future credit starts,True
7,No future actual end dates,True
8,No future credit updates,True
9,High-missing AMT_ANNUITY removed,True


All checks passed.


## Save the cleaned bureau data

In [31]:
cleaning_audit = pd.DataFrame([
    {"rule": "Out-of-scope bureau records removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicate rows removed", "affected": exact_duplicates},
    {"rule": "Rare credit types consolidated", "affected": int(bureau_clean["CREDIT_TYPE"].eq("Other rare credit").sum())},
    {"rule": "Date anomaly records flagged", "affected": int(bureau_clean["BUREAU_DATE_ANOMALY"].sum())},
    {"rule": "Features removed by missingness policy", "affected": len(removed_features)},
])
display(cleaning_audit)
bureau_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "bureau_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "bureau_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "bureau_cleaning_validation.csv", index=False)

print("Clean bureau dataset saved:", output_path)
print("Output rows:", len(bureau_clean))
print("Output columns:", bureau_clean.shape[1])
print("Unique applicants:", bureau_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(bureau_clean.select_dtypes(include="number").isna().sum().sum()))

,rule,affected
0,Out-of-scope bureau records removed,251103
1,Exact duplicate rows removed,0
2,Rare credit types consolidated,69
3,Date anomaly records flagged,247
4,Features removed by missingness policy,2


Clean bureau dataset saved: /Users/taranveersingh/A-MRP/data/interim/bureau_clean.pkl
Output rows: 1465325
Output columns: 22
Unique applicants: 263491
Remaining numerical missing values: 1346785


## Main cleaning results

The cleaned bureau data contains 1,465,325 credit records for 263,491 applicants. All bureau IDs are complete and unique.

Rare currencies and credit types were grouped into simpler categories. Invalid future and extreme date values were changed to missing, while the original records were kept.

Unusual financial values were also kept and marked with new indicator columns. This avoids removing records that may represent real overpayments, corrections or reporting differences.

Two features were removed because more than half of their training-linked values were missing. The cleaned bureau data has 22 columns. The next step is to clean the previous-application data.